# Forecast v2 GPU - XGBoost CUDA + LightGBM Ensemble

This notebook replaces the previous `src/09_forecast_ver2.py` script. It runs directly as an `.ipynb` and is configured for an NVIDIA RTX 3050 6GB VRAM.

Training rule requested:

- Train = old train + old valid: `2012-07-04 -> 2019-12-31`
- Test benchmark = `2020-01-01 -> 2022-12-31`
- No hyperparameter tuning on test
- Outputs are written to `outputs/forecast_ver2/`

Model v2:

- XGBoost GPU (`device="cuda"`) as primary model
- LightGBM CPU as secondary model
- Fixed weighted blend, selected before looking at test metrics
- Separate direct models for `Revenue` and `COGS`
- Additional COGS ratio model to stabilize COGS level
- No forced `COGS < Revenue`, because historical data contains days where `COGS >= Revenue`

In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import json
import os
import random
import warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent

DATASET_DIR = ROOT / "dataset"
OUT_DIR = ROOT / "outputs" / "forecast_ver2"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
MODEL_DIR = OUT_DIR / "models"
for p in [OUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATE_COL = "Date"
TARGETS = ["Revenue", "COGS"]
print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# GPU availability check
import subprocess
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    print(result.stdout.splitlines()[0] if result.stdout else "nvidia-smi returned no output")
except Exception as exc:
    print("nvidia-smi unavailable:", exc)

from xgboost import XGBRegressor
import lightgbm as lgb

# Verify XGBoost CUDA with a tiny fit. If this fails, set DEVICE='cpu'.
DEVICE = "cuda"
try:
    X_small = np.random.default_rng(SEED).normal(size=(512, 8))
    y_small = X_small[:, 0] + np.random.default_rng(SEED + 1).normal(size=512)
    gpu_probe = XGBRegressor(n_estimators=3, max_depth=2, tree_method="hist", device="cuda", random_state=SEED)
    gpu_probe.fit(X_small, y_small)
    print("XGBoost CUDA: OK")
except Exception as exc:
    DEVICE = "cpu"
    print("XGBoost CUDA failed; using CPU fallback:", repr(exc))
print("XGBoost device:", DEVICE)

## 1. Load Dataset

Uses the processed core Parquet files already built in `dataset/`.

In [ ]:
train = pd.read_parquet(DATASET_DIR / "model_core_train.parquet")
valid = pd.read_parquet(DATASET_DIR / "model_core_valid.parquet")
test = pd.read_parquet(DATASET_DIR / "model_core_test.parquet")
history = pd.read_parquet(DATASET_DIR / "model_core_history.parquet")
future_template = pd.read_parquet(DATASET_DIR / "model_core_future.parquet")

for df in [train, valid, test, history, future_template]:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

train_full = pd.concat([train, valid], ignore_index=True).sort_values(DATE_COL).reset_index(drop=True)
FEATURE_COLS = [c for c in train_full.columns if c not in [DATE_COL, "Revenue", "COGS"]]

print(f"train_full: {len(train_full):,} | {train_full.Date.min().date()} -> {train_full.Date.max().date()}")
print(f"test      : {len(test):,} | {test.Date.min().date()} -> {test.Date.max().date()}")
print(f"future    : {len(future_template):,} | {future_template.Date.min().date()} -> {future_template.Date.max().date()}")
print("features  :", len(FEATURE_COLS))
print("Historical COGS >= Revenue rate:", float((history.COGS >= history.Revenue).mean()))

## 2. Model Helpers

In [ ]:
def metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "R2": float(r2_score(y_true, pred)),
    }


def print_metrics(label, target, m):
    print(f"[{label} | {target}] MAE={m['MAE']:.2f} RMSE={m['RMSE']:.2f} R2={m['R2']:.4f}")


def make_xgb(seed=SEED, n_estimators=900):
    return XGBRegressor(
        n_estimators=n_estimators,
        max_depth=4,
        learning_rate=0.035,
        subsample=0.88,
        colsample_bytree=0.88,
        min_child_weight=4,
        reg_alpha=0.05,
        reg_lambda=0.8,
        objective="reg:squarederror",
        tree_method="hist",
        device=DEVICE,
        random_state=seed,
        n_jobs=0,
        verbosity=0,
    )


def make_lgb(seed=SEED, n_estimators=900):
    return lgb.LGBMRegressor(
        objective="regression",
        metric="rmse",
        n_estimators=n_estimators,
        learning_rate=0.035,
        num_leaves=31,
        min_child_samples=18,
        subsample=0.88,
        subsample_freq=1,
        colsample_bytree=0.88,
        reg_alpha=0.05,
        reg_lambda=0.8,
        random_state=seed,
        seed=seed,
        deterministic=True,
        force_col_wise=True,
        verbosity=-1,
    )


def fit_log_model(model, X, y):
    model.fit(X, np.log1p(y))
    return model


def predict_log_model(model, X):
    return np.expm1(model.predict(X)).clip(min=0)


def fit_raw_model(model, X, y):
    model.fit(X, y)
    return model


def weighted_blend(pred_xgb, pred_lgb, wx=0.58):
    return (wx * pred_xgb + (1 - wx) * pred_lgb).clip(min=0)

## 3. Train on Train+Valid and Benchmark on Test

In [ ]:
X_train = train_full[FEATURE_COLS]
X_test = test[FEATURE_COLS]

models = {}
preds = {}
metric_rows = []

# Revenue direct models
models["revenue_xgb"] = fit_log_model(make_xgb(SEED), X_train, train_full["Revenue"])
models["revenue_lgb"] = fit_log_model(make_lgb(SEED), X_train, train_full["Revenue"])
preds["Revenue_xgb"] = predict_log_model(models["revenue_xgb"], X_test)
preds["Revenue_lgb"] = predict_log_model(models["revenue_lgb"], X_test)
preds["Revenue_blend"] = weighted_blend(preds["Revenue_xgb"], preds["Revenue_lgb"], wx=0.58)
preds["Revenue_final"] = preds["Revenue_xgb"]  # GPU primary is final v2; LGBM is kept for comparison.

# COGS direct models
models["cogs_xgb"] = fit_log_model(make_xgb(SEED + 10), X_train, train_full["COGS"])
models["cogs_lgb"] = fit_log_model(make_lgb(SEED + 10), X_train, train_full["COGS"])
preds["COGS_xgb_direct"] = predict_log_model(models["cogs_xgb"], X_test)
preds["COGS_lgb_direct"] = predict_log_model(models["cogs_lgb"], X_test)
preds["COGS_direct_blend"] = weighted_blend(preds["COGS_xgb_direct"], preds["COGS_lgb_direct"], wx=0.58)

# COGS ratio models: helps the COGS scale follow Revenue without hard-capping COGS < Revenue.
ratio_train = (train_full["COGS"] / train_full["Revenue"]).replace([np.inf, -np.inf], np.nan)
ratio_train = ratio_train.fillna(ratio_train.median()).clip(0.35, 1.45)
models["ratio_xgb"] = fit_raw_model(make_xgb(SEED + 20, n_estimators=700), X_train, ratio_train)
models["ratio_lgb"] = fit_raw_model(make_lgb(SEED + 20, n_estimators=700), X_train, ratio_train)
ratio_xgb = models["ratio_xgb"].predict(X_test).clip(0.35, 1.45)
ratio_lgb = models["ratio_lgb"].predict(X_test).clip(0.35, 1.45)
preds["COGS_ratio_based"] = preds["Revenue_blend"] * weighted_blend(ratio_xgb, ratio_lgb, wx=0.58)
preds["COGS_blend"] = (0.75 * preds["COGS_direct_blend"] + 0.25 * preds["COGS_ratio_based"]).clip(min=0)
preds["COGS_final"] = preds["COGS_xgb_direct"]  # GPU primary is final v2; ratio model is kept for diagnostics.

for target in TARGETS:
    truth = test[target].to_numpy()
    target_pred_keys = [k for k in preds if k.startswith(target)]
    for key in target_pred_keys:
        m = metrics(truth, preds[key])
        metric_rows.append({"split": "test", "target": target, "model": key, **m})
        print_metrics("test", key, m)

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(TABLE_DIR / "metrics_ver2_gpu.csv", index=False)
metrics_df

In [ ]:
test_predictions = test[[DATE_COL, "Revenue", "COGS"]].copy()
test_predictions["Revenue_pred"] = preds["Revenue_final"]
test_predictions["COGS_pred"] = preds["COGS_final"]
test_predictions.to_csv(TABLE_DIR / "test_predictions_ver2_gpu.csv", index=False)
test_predictions.head()

## 4. Feature Importance

In [ ]:
def save_importance(model, label):
    importance = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
    importance.to_csv(TABLE_DIR / f"feature_importance_{label}.csv", index=False)
    top = importance.head(25).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"], top["importance"])
    plt.title(f"Feature Importance - {label}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"feature_importance_{label}.png", dpi=160)
    plt.close()
    return importance

imp_revenue_xgb = save_importance(models["revenue_xgb"], "revenue_xgb")
imp_cogs_xgb = save_importance(models["cogs_xgb"], "cogs_xgb")
display(imp_revenue_xgb.head(20))

## 5. Refit on Full History and Forecast Future

In [ ]:
# Refit models on all observed history through 2022-12-31 before forecasting future.
X_hist = history[FEATURE_COLS]
final_models = {}
final_models["revenue_xgb"] = fit_log_model(make_xgb(SEED), X_hist, history["Revenue"])
final_models["cogs_xgb"] = fit_log_model(make_xgb(SEED + 10), X_hist, history["COGS"])
ratio_hist = (history["COGS"] / history["Revenue"]).replace([np.inf, -np.inf], np.nan)
ratio_hist = ratio_hist.fillna(ratio_hist.median()).clip(0.35, 1.45)
final_models["ratio_xgb"] = fit_raw_model(make_xgb(SEED + 20, n_estimators=700), X_hist, ratio_hist)

for name, model in final_models.items():
    joblib.dump(model, MODEL_DIR / f"{name}.joblib")
print("Saved models:", MODEL_DIR)

In [ ]:
# Recursive feature builder for future dates. Uses src/08_build_modeling_dataset.py functions.
import importlib.util
spec = importlib.util.spec_from_file_location("feature_builder", ROOT / "src" / "08_build_modeling_dataset.py")
FB = importlib.util.module_from_spec(spec)
spec.loader.exec_module(FB)


def make_seasonal_lookup(real_history):
    temp = real_history[[DATE_COL, "Revenue", "COGS"]].copy()
    temp["season_month"] = temp[DATE_COL].dt.month
    temp["season_day"] = temp[DATE_COL].dt.day
    lookup = {}
    global_stats = temp[TARGETS].agg(["mean", "std"]).to_dict()
    for target in TARGETS:
        stats = temp.groupby(["season_month", "season_day"])[target].agg(["mean", "std"])
        lookup[target] = {"stats": stats, "global": global_stats[target]}
    return lookup


def add_fixed_seasonal_priors(row, seasonal_lookup):
    month = int(row["month"].iloc[0])
    day = int(row["dayofmonth"].iloc[0])
    for target in TARGETS:
        stats = seasonal_lookup[target]["stats"]
        global_stats = seasonal_lookup[target]["global"]
        if (month, day) in stats.index:
            mean_value = stats.loc[(month, day), "mean"]
            std_value = stats.loc[(month, day), "std"]
        else:
            mean_value = global_stats["mean"]
            std_value = global_stats["std"]
        row[f"{target}_md_prior_mean"] = mean_value if pd.notna(mean_value) else global_stats["mean"]
        row[f"{target}_md_prior_std"] = std_value if pd.notna(std_value) else global_stats["std"]
    return row


def make_recursive_row(buffer, date, feature_cols, seasonal_lookup, base_year):
    # Max lag/rolling horizon is 365, so tail 500 is enough and keeps this fast.
    buffer = buffer.tail(500).copy()
    tmp = pd.concat([
        buffer[[DATE_COL, "Revenue", "COGS"]],
        pd.DataFrame({DATE_COL: [date], "Revenue": [np.nan], "COGS": [np.nan]}),
    ], ignore_index=True).sort_values(DATE_COL).reset_index(drop=True)
    tmp = FB.add_calendar_features(tmp, base_year=base_year)
    tmp = FB.add_holiday_features(tmp)
    tmp = FB.add_target_lag_features(tmp)
    row = tmp[tmp[DATE_COL].eq(date)].tail(1).copy()
    row = add_fixed_seasonal_priors(row, seasonal_lookup)
    for col in feature_cols:
        if col not in row.columns:
            row[col] = np.nan
    return row[feature_cols]


def predict_future_recursive(real_history, future_dates):
    buffer = real_history[[DATE_COL, "Revenue", "COGS"]].copy().sort_values(DATE_COL).reset_index(drop=True)
    base_year = int(buffer[DATE_COL].dt.year.min())
    seasonal_lookup = make_seasonal_lookup(buffer)
    rows = []
    for date in pd.to_datetime(future_dates):
        X_row = make_recursive_row(buffer, date, FEATURE_COLS, seasonal_lookup, base_year)
        rev_xgb = predict_log_model(final_models["revenue_xgb"], X_row)[0]
        revenue = float(rev_xgb)

        cogs_xgb = predict_log_model(final_models["cogs_xgb"], X_row)[0]
        cogs_direct = float(cogs_xgb)
        ratio_xgb = float(final_models["ratio_xgb"].predict(X_row)[0])
        ratio = float(ratio_xgb)
        ratio = float(np.clip(ratio, 0.35, 1.45))
        cogs_ratio_based = revenue * ratio
        cogs = max(0.0, cogs_direct)

        rows.append({DATE_COL: date, "Revenue": revenue, "COGS": cogs})
        buffer = pd.concat([buffer, pd.DataFrame([{DATE_COL: date, "Revenue": revenue, "COGS": cogs}])], ignore_index=True)
    return pd.DataFrame(rows)

future_pred = predict_future_recursive(history, future_template[DATE_COL])
submission = future_pred.copy()
submission["Revenue"] = submission["Revenue"].clip(lower=0).round(2)
submission["COGS"] = submission["COGS"].clip(lower=0).round(2)
submission_out = submission.copy()
submission_out[DATE_COL] = submission_out[DATE_COL].dt.strftime("%Y-%m-%d")
submission_out.to_csv(OUT_DIR / "submission_ver2_gpu.csv", index=False)
submission_out.head(10)

## 6. Plots and Summary

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(test[DATE_COL], test["Revenue"], label="Actual Revenue", linewidth=0.8)
plt.plot(test_predictions[DATE_COL], test_predictions["Revenue_pred"], label="Predicted Revenue v2", linewidth=0.8)
plt.title("Forecast v2 GPU - Test Revenue Benchmark")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "test_revenue_prediction_ver2_gpu.png", dpi=160)
plt.close()

plt.figure(figsize=(15, 5))
plt.plot(history[DATE_COL], history["Revenue"], label="Historical Revenue", linewidth=0.7)
plt.plot(submission[DATE_COL], submission["Revenue"], label="Future Forecast v2 GPU", linewidth=0.9)
plt.title("Forecast v2 GPU - Future Revenue")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "future_revenue_forecast_ver2_gpu.png", dpi=160)
plt.show()

In [ ]:
summary = {
    "device": DEVICE,
    "train_rows": int(len(train_full)),
    "test_rows": int(len(test)),
    "feature_count": int(len(FEATURE_COLS)),
    "test_metrics": metrics_df.to_dict(orient="records"),
    "outputs": {
        "submission": str(OUT_DIR / "submission_ver2_gpu.csv"),
        "metrics": str(TABLE_DIR / "metrics_ver2_gpu.csv"),
        "test_predictions": str(TABLE_DIR / "test_predictions_ver2_gpu.csv"),
    },
}
(OUT_DIR / "run_summary_ver2_gpu.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2)[:4000])

In [ ]:
print("Output files:")
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(ROOT))